# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shwetabh1013/flyrank-ml-internship-shwetabh/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — the model W06 selected

W06 picked **logistic regression on time-honest features** (selection rule: best mean precision@20 on 25 client-grouped folds; measured 0.828 ± 0.091 vs 0.633 base rate). Here I fit it once on **all** measurable pages and use it to build the queue.

**How "train on the past, score the present" works with one snapshot:**
- **Training** uses features as of 30 days ago (days 31–90) and the label from days 1–30 — the outcome the model learns to anticipate.
- **Scoring** shifts the same features forward by 30 days (days 1–60) and asks: *which pages look like the ones that declined last time?* That is a genuine forecast of the **next** 30 days — no label exists for it yet.
- Assumption this rests on: the relationship between these signals and a decline is similar from one month to the next. Section 4 monitors exactly that.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append(str(Path("..").resolve()))
import refresh_lib as rl
warnings.filterwarnings("ignore", category=FutureWarning)
OUT = Path("../outputs"); OUT.mkdir(exist_ok=True)

df = rl.load_data()
train = rl.measurable(df)                                   # >= 30 impressions in days 31-60
X_train = rl.honest_features(train, "T-30")
model = rl.make_lr().fit(X_train, train["y"])

# score the present: pages with >= 30 impressions in the last 30 days (the next label's denominator)
live = df[df["impressions_last_30d"] >= rl.MEASURABLE_FLOOR].copy()
X_live = rl.honest_features(live, "now").reindex(columns=X_train.columns, fill_value=0)
live["risk"] = model.predict_proba(X_live)[:, 1]
# class_weight="balanced" shifts probabilities -> use ranks, not the raw number
live["risk_pct_in_client"] = live.groupby("client_id")["risk"].rank(pct=True)

print(f"trained on {len(train):,} pages / {train['client_id'].nunique()} clients")
print(f"scoring   {len(live):,} pages / {live['client_id'].nunique()} clients")

coefs = pd.Series(model[-1].coef_[0], index=X_train.columns).sort_values()
coefs.round(3)

trained on 21,578 pages / 31 clients
scoring   20,128 pages / 29 clients


content_age_days                  -0.396
log_clicks_recent                 -0.363
log_sessions_recent               -0.229
log_clicks_before                 -0.156
ctr_60d                           -0.064
search_volume                     -0.032
main_intent_commercial            -0.028
word_count                        -0.024
main_intent_informational         -0.022
content_type_comparison article   -0.012
cpc                               -0.004
main_intent_transactional          0.003
main_intent_navigational           0.003
content_type_feedly article        0.005
content_type_keyword article       0.005
competition                        0.013
impression_momentum                0.023
has_keyword_data                   0.042
main_intent_unknown                0.136
log_sessions_before                0.151
has_word_count                     0.167
log_impressions_before             0.189
log_impressions_recent             0.215
dtype: float64

**What the model leans on (standardised coefficients; positive = more decline risk):**
- **Visible but not clicked:** recent impressions push risk *up*, recent clicks and sessions push it *down*. Same visibility, fewer clicks → higher risk. This is the CTR-gap intuition from my W04 rule — the data keeps that part.
- **Younger pages** carry more risk than older ones. The W04 "stale = at risk" idea is not supported: staleness had no signal in W06 (AUC ≈ 0.49).
- **Momentum** (recent vs earlier month) is near zero — a page that just rose is not more likely to fall back.
- Impressions and clicks are strongly correlated, so read their coefficients as a *pair* (impressions-per-click), never one at a time.

## 1. Ranked actions + reason codes

**Reason codes are built from the model itself**, so they explain the actual score rather than a separate story. Each page's score is a sum of `coefficient × standardised value`; I add those contributions up into five plain-language groups and show the two that push risk up the most.

| Code | Plain words | Built from |
|---|---|---|
| `VISIBLE_LOW_CLICKS` | Earning impressions but few clicks for that visibility | impressions, clicks, CTR group pushes risk up and CTR is below the client's median |
| `HIGH_VISIBILITY` | A lot of recent search exposure to lose | same group, CTR not below median |
| `LOW_SESSIONS` | Few sessions relative to the portfolio | sessions group |
| `YOUNGER_PAGE` | Younger than typical — still settling in search | age |
| `PROFILE` | Content/keyword profile resembles past decliners | word count, keyword data, type, intent |

**The queue:** one per client (an editor works one site at a time), top 10 by risk. **Action** comes from risk × impact:
- `REVIEW_FIRST` — top-10 risk **and** recent impressions at or above the client's median: most traffic at stake.
- `CTR_CHECK` — top-10 risk, mainly `VISIBLE_LOW_CLICKS`, below-median traffic: title/meta check, cheap.
- `WATCH` — top-10 risk but small traffic: re-check next month before spending editor time.

In [2]:
scaler, lr = model[0], model[-1]
z = (X_live - scaler.mean_) / scaler.scale_
contrib = z * lr.coef_[0]

groups = {
    "traffic":  ["log_impressions_recent", "log_impressions_before", "log_clicks_recent",
                 "log_clicks_before", "ctr_60d", "impression_momentum"],
    "sessions": ["log_sessions_recent", "log_sessions_before"],
    "age":      ["content_age_days"],
}
groups["profile"] = [c for c in X_live.columns if c not in sum(groups.values(), [])]
g = pd.DataFrame({k: contrib[v].sum(axis=1) for k, v in groups.items()}, index=live.index)

client_ctr_median = live.groupby("client_id")["ctr"].transform("median")
def code_for(group, row_idx):
    if group == "traffic":
        return "VISIBLE_LOW_CLICKS" if live.at[row_idx, "ctr"] < client_ctr_median.at[row_idx] else "HIGH_VISIBILITY"
    return {"sessions": "LOW_SESSIONS", "age": "YOUNGER_PAGE", "profile": "PROFILE"}[group]

def reason_codes(row_idx):
    top = g.loc[row_idx].sort_values(ascending=False)
    top = top[top > 0].head(2)
    return "|".join(code_for(k, row_idx) for k in top.index) or "NONE"

live["queue_rank"] = live.groupby("client_id")["risk"].rank(ascending=False, method="first").astype(int)
queue = live[live["queue_rank"] <= 10].copy()
queue["reason_codes"] = [reason_codes(i) for i in queue.index]
imp_median = live.groupby("client_id")["impressions_last_30d"].transform("median")
queue["action"] = np.select(
    [queue["impressions_last_30d"] >= imp_median.loc[queue.index],
     queue["reason_codes"].str.startswith("VISIBLE_LOW_CLICKS")],
    ["REVIEW_FIRST", "CTR_CHECK"], default="WATCH")
queue = queue.sort_values(["client_id", "queue_rank"])

show = ["client_id", "queue_rank", "content_id", "action", "reason_codes",
        "risk_pct_in_client", "impressions_last_30d", "clicks_last_30d", "content_age_days"]
biggest_client = live["client_id"].value_counts().index[0]
print(f"{queue['client_id'].nunique()} client queues, {len(queue)} pages total. Example: the largest client's queue")
queue[queue["client_id"] == biggest_client][show].round(3)

29 client queues, 280 pages total. Example: the largest client's queue


,client_id,queue_rank,content_id,action,reason_codes,risk_pct_in_client,impressions_last_30d,clicks_last_30d,content_age_days
26081,client_19581e27de,1,content_245531f14938,REVIEW_FIRST,VISIBLE_LOW_CLICKS|YOUNGER_PAGE,1.000,821,0,95
23897,client_19581e27de,2,content_d6e535ef361f,REVIEW_FIRST,HIGH_VISIBILITY|LOW_SESSIONS,1.000,1092,0,112
18051,client_19581e27de,3,content_4d7bba3dcb7c,REVIEW_FIRST,VISIBLE_LOW_CLICKS|YOUNGER_PAGE,1.000,2666,1,103
26438,client_19581e27de,4,content_73df81553583,REVIEW_FIRST,VISIBLE_LOW_CLICKS|YOUNGER_PAGE,1.000,1547,0,103
20832,client_19581e27de,5,content_fa3aad50d0be,REVIEW_FIRST,VISIBLE_LOW_CLICKS|YOUNGER_PAGE,0.999,519,0,98
29774,client_19581e27de,6,content_b1b25cc9ee37,REVIEW_FIRST,VISIBLE_LOW_CLICKS|YOUNGER_PAGE,0.999,1171,0,98
1885,client_19581e27de,7,content_33263b66aacf,REVIEW_FIRST,VISIBLE_LOW_CLICKS|YOUNGER_PAGE,0.999,779,0,98
14186,client_19581e27de,8,content_12b8d8a8d62f,REVIEW_FIRST,VISIBLE_LOW_CLICKS|YOUNGER_PAGE,0.999,5692,1,98
28527,client_19581e27de,9,content_af33d56338a4,WATCH,YOUNGER_PAGE|HIGH_VISIBILITY,0.999,459,0,95
6964,client_19581e27de,10,content_c9c8fd0e7d30,REVIEW_FIRST,VISIBLE_LOW_CLICKS|YOUNGER_PAGE,0.999,577,0,119


In [3]:
print("Action mix across all queues:")
print(queue["action"].value_counts().to_string(), "\n")
print("Lead reason code:")
print(queue["reason_codes"].str.split("|").str[0].value_counts().to_string())

Action mix across all queues:
action
REVIEW_FIRST    180
WATCH            93
CTR_CHECK         7 

Lead reason code:
reason_codes
PROFILE               94
VISIBLE_LOW_CLICKS    80
YOUNGER_PAGE          61
HIGH_VISIBILITY       32
LOW_SESSIONS          13


**Read:**
- **29 client queues, 280 pages.** 180 are `REVIEW_FIRST` (real traffic at stake), 93 `WATCH`, 7 `CTR_CHECK`.
- **The largest client's queue is a clear pattern:** pages ~95–120 days old, hundreds to thousands of monthly impressions, and **0–1 clicks**. That matches the model's two strongest signals (younger + visible-but-unclicked). An editor can verify each one in a minute: is the title/snippet matching what people search?
- **`PROFILE` is the most common lead code (94 pages) and the weakest one.** It means the score comes mainly from metadata patterns (e.g. unknown search intent, which content type it is) rather than the page's traffic. Treat `PROFILE`-led pages as lower confidence: check them after the traffic-led ones.

## 2. Intended use and limits

**Who:** a FlyRank editor or content strategist planning the next month's refresh work for one client site.

**For what:** deciding which ~10 pages to *look at first*. The queue says "these pages resemble pages that lost >20% of impressions last month" — it is a review list, not a to-do list.

**Measured quality (W06, client-grouped, pages with ≥ 30 impressions/month):** precision@20 0.828 ± 0.091 vs 0.633 base rate; per-client precision@10 0.758 vs ~0.61 for random ranking. Out of 10 pages in a queue, expect roughly 7–8 to actually decline, versus 6 if you picked at random. A real, modest lift.

**Where it stops being valid:**
- **Pages under 30 impressions a month** — not scored. A "decline" there is noise.
- **Whole-site ranking.** AUC ≈ 0.62: below the top of the queue, the order means little. Don't read rank 300 vs 400 as anything.
- **New clients with no history**, or clients whose tracking just changed.
- **Effect of refreshing.** Nothing here measures whether a refresh *stops* a decline. Pages were not refreshed at random, so the data can't answer it.
- **The future.** Trained on one 90-day snapshot from one period. Seasonality, algorithm updates and tracking changes are not represented.
- **Probabilities.** Class weighting shifts them; only within-client ranks are used.

In [4]:
print("Coverage of the live queue:")
print(f"  scored pages: {len(live):,} of {len(df):,} ({len(live)/len(df):.0%}); the rest are below the traffic floor")
print(f"  clients with a queue: {queue['client_id'].nunique()} of {df['client_id'].nunique()}")
small = live.groupby("client_id").size()
print(f"  clients with < 10 scorable pages (queue shorter than 10): {(small < 10).sum()}")

Coverage of the live queue:
  scored pages: 20,128 of 30,000 (67%); the rest are below the traffic floor
  clients with a queue: 29 of 32
  clients with < 10 scorable pages (queue shorter than 10): 3


## 3. Human review + the no-go list

**A person must check, for every page before acting:**
1. **Is the decline real?** Open Search Console for the page: is the drop steady, or one bad day / a tracking gap?
2. **Is it seasonal?** Compare with the same month last year where history exists. A Christmas page falling in January is not a refresh candidate.
3. **Did the query mix change?** If the page lost a query it never served well, the fix may be a different page, not this one.
4. **Was it just updated?** 68% of pages show an update in the last 30 days (mostly batch dates). If a real edit just happened, wait for it to settle.
5. **Does the reason code make sense on the page?** `VISIBLE_LOW_CLICKS` → look at title and meta first; `YOUNGER_PAGE` → check if it's still finding its rankings before rewriting.

**Never automate:**
- **Deleting, noindexing, merging or redirecting** pages from this score. A model flag is not a reason to remove content.
- **Auto-rewriting** flagged pages with an LLM and publishing without review.
- **Client-facing claims** like "these pages will lose traffic" or "refreshing will recover X clicks". The score supports *a review decision*; it measures neither.
- **Ranking clients against each other** with it — client base rates differ a lot (W06 std ≈ 0.10), and the model was validated within held-out clients, not across them.

In [5]:
no_go = pd.DataFrame({
    "action": ["delete / noindex / redirect", "auto-rewrite + publish", "promise traffic outcomes",
               "compare clients by score", "score pages < 30 impressions/month"],
    "status": "never automated",
    "why": ["irreversible; score is decision support only", "no human check of quality or intent",
            "no causal design; observational data", "validated within clients, not across",
            "label is noise at that volume"],
})
no_go

,action,status,why
0,delete / noindex / redirect,never automated,irreversible; score is decision support only
1,auto-rewrite + publish,never automated,no human check of quality or intent
2,promise traffic outcomes,never automated,no causal design; observational data
3,compare clients by score,never automated,"validated within clients, not across"
4,score pages < 30 impressions/month,never automated,label is noise at that volume


## 4. Monitoring / retrain triggers

**Check monthly, once the next 30 days of data arrive:**
1. **Hit rate of last month's queue.** For each client, what share of its top 10 actually fell >20%? If the mean drops below that client set's base rate for **two months running**, the model is no better than random → retrain or retire.
2. **Feature drift (PSI).** Population Stability Index between the features the model was trained on and the features it is scoring. PSI > 0.2 on a feature = it looks different enough that the coefficients may not hold. Computed below for *this* run (training at T-30 vs scoring now).
3. **Pipeline changes that silently break inputs:** a new batch-update date (like the 20/104-day clusters), GA4 tracking gaps (sessions drop to zero client-wide), new content types, or a client added/removed.
4. **Scheduled retrain** every month on the newest window regardless — one snapshot is a thin basis.

In [6]:
def psi(expected, actual, bins=10):
    cuts = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    if len(cuts) < 3:
        return np.nan
    e = np.histogram(expected, cuts)[0] / len(expected)
    a = np.histogram(np.clip(actual, cuts[0], cuts[-1]), cuts)[0] / len(actual)
    e, a = np.clip(e, 1e-4, None), np.clip(a, 1e-4, None)
    return float(np.sum((a - e) * np.log(a / e)))

numeric = ["log_impressions_recent", "log_impressions_before", "impression_momentum",
           "log_clicks_recent", "log_clicks_before", "ctr_60d", "log_sessions_recent",
           "log_sessions_before", "content_age_days"]
drift = pd.Series({c: psi(X_train[c], X_live[c]) for c in numeric}).round(3).sort_values(ascending=False)
drift_table = pd.DataFrame({"psi": drift,
                            "status": np.where(drift > 0.2, "RETRAIN CHECK", np.where(drift > 0.1, "watch", "ok"))})
drift_table

,psi,status
content_age_days,0.926,RETRAIN CHECK
log_sessions_before,0.098,ok
log_sessions_recent,0.084,ok
impression_momentum,0.022,ok
log_impressions_recent,0.015,ok
ctr_60d,0.012,ok
log_impressions_before,0.009,ok
log_clicks_recent,0.005,ok
log_clicks_before,0.003,ok


**Read:**
- **Traffic features are stable** (PSI ≤ 0.1): the scoring month looks like the training month on impressions, clicks, CTR and momentum.
- **`content_age_days` flags (PSI ≈ 0.93) — and that is by construction, not a surprise.** Every page is exactly 30 days older at scoring time, so the youngest pages now are 90 days old instead of 60. The model learned "younger = riskier" partly from 60–90-day-old pages it will never see again in this snapshot. The live queue is dominated by the youngest pages (~95–120 days), so this is the assumption most worth testing in the first monthly hit-rate check.
- Practical consequence: **retrain monthly on the newest window**, and if the hit rate on young-page-led queue items falls below base rate, drop or cap the age feature.

## 5. Exports for the paper

Written to `work/outputs/` (CSV files there are git-ignored — the repo never ships data; re-run this notebook to regenerate). The paper embeds only aggregates and a 10-row sample with pseudonymous ids.

In [7]:
queue[show + ["risk"]].to_csv(OUT / "w07_queue_top10_per_client.csv", index=False)
live[["client_id", "content_id", "risk", "risk_pct_in_client", "impressions_last_30d"]].to_csv(
    OUT / "w07_scores_all_pages.csv", index=False)
coefs.rename("coefficient").to_csv(OUT / "w07_model_coefficients.csv")
drift_table.to_csv(OUT / "w07_feature_drift.csv")
summary = {
    "pages_trained": int(len(train)), "pages_scored": int(len(live)),
    "client_queues": int(queue["client_id"].nunique()), "queue_pages": int(len(queue)),
    "action_mix": queue["action"].value_counts().to_dict(),
    "lead_reason_mix": queue["reason_codes"].str.split("|").str[0].value_counts().to_dict(),
}
pd.Series(summary).to_json(OUT / "w07_summary.json", indent=2)
for f in sorted(OUT.glob("w07_*")):
    print(f.name)

w07_feature_drift.csv
w07_model_coefficients.csv
w07_queue_top10_per_client.csv
w07_scores_all_pages.csv
w07_summary.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.